In [9]:
import time
from smbus2 import SMBus

def diagnose_zed_f9p(i2c_bus=1, address=0x42):
    """Comprehensive diagnostic for ZED-F9P I2C connection."""
    
    print("=" * 60)
    print("ZED-F9P I2C DIAGNOSTIC")
    print("=" * 60)
    
    # Test 1: Check I2C bus accessibility
    print("\n[1] Testing I2C Bus Access...")
    try:
        bus = SMBus(i2c_bus)
        print(f"✓ I2C Bus {i2c_bus} opened successfully")
    except Exception as e:
        print(f"✗ Failed to open I2C bus: {e}")
        return
    
    # Test 2: Check device presence
    print(f"\n[2] Testing Device Presence at {hex(address)}...")
    try:
        # Try to read a byte to see if device responds
        bus.read_byte(address)
        print(f"✓ Device responds at address {hex(address)}")
    except Exception as e:
        print(f"✗ No response from device: {e}")
        print("\nTroubleshooting:")
        print("  - Run 'i2cdetect -y 1' to scan for devices")
        print("  - Check if device is at 0x42 (default)")
        print("  - Verify wiring: SDA, SCL, GND, VCC")
        bus.close()
        return
    
    # Test 3: Check data stream registers
    print("\n[3] Reading Data Stream Registers...")
    try:
        high_byte = bus.read_byte_data(address, 0xFD)
        low_byte = bus.read_byte_data(address, 0xFE)
        bytes_available = (high_byte << 8) | low_byte
        print(f"✓ Registers readable")
        print(f"  Bytes available: {bytes_available}")
        
        if bytes_available == 0xFFFF:
            print("  ⚠ Warning: 0xFFFF might indicate register issue")
        elif bytes_available == 0:
            print("  ⚠ No data in buffer (might be normal if just started)")
    except Exception as e:
        print(f"✗ Cannot read registers: {e}")
        bus.close()
        return
    
    # Test 4: Monitor data stream for 10 seconds
    print("\n[4] Monitoring Data Stream (10 seconds)...")
    print("Timestamp | Bytes Available | Raw Data Preview")
    print("-" * 60)
    
    data_seen = False
    for i in range(20):  # 20 checks over 10 seconds
        try:
            high_byte = bus.read_byte_data(address, 0xFD)
            low_byte = bus.read_byte_data(address, 0xFE)
            bytes_available = (high_byte << 8) | low_byte
            
            if bytes_available > 0 and bytes_available < 0xFFFF:
                data_seen = True
                chunk_size = min(bytes_available, 32)
                raw_data = bus.read_i2c_block_data(address, 0xFF, chunk_size)
                preview = bytes(raw_data[:20]).decode('ascii', errors='ignore')
                print(f"{i*0.5:04.1f}s    | {bytes_available:5d}          | {preview}")
            
            time.sleep(0.5)
        except Exception as e:
            print(f"Error during monitoring: {e}")
            break
    
    if not data_seen:
        print("\n✗ No data received from GPS module")
        print("\nPossible issues:")
        print("  1. GPS module not configured for NMEA output on I2C")
        print("  2. Module still acquiring satellite fix (can take minutes)")
        print("  3. Antenna not connected or faulty")
        print("  4. Module needs configuration via U-Center")
    else:
        print("\n✓ Data stream is active")
    
    # Test 5: Check for NMEA sentences
    print("\n[5] Checking for NMEA Sentences...")
    buffer = bytearray()
    nmea_found = False
    
    for i in range(10):
        try:
            high_byte = bus.read_byte_data(address, 0xFD)
            low_byte = bus.read_byte_data(address, 0xFE)
            bytes_available = (high_byte << 8) | low_byte
            
            if 0 < bytes_available < 0xFFFF:
                chunk_size = min(bytes_available, 32)
                raw_data = bus.read_i2c_block_data(address, 0xFF, chunk_size)
                buffer.extend(raw_data)
                
                # Look for NMEA sentence markers
                if b'$G' in buffer or b'GGA' in buffer or b'RMC' in buffer:
                    nmea_found = True
                    # Try to extract a complete sentence
                    if b'\r\n' in buffer:
                        line_end = buffer.index(b'\r\n')
                        if b'$' in buffer[:line_end]:
                            start = buffer.index(b'$')
                            sentence = buffer[start:line_end].decode('ascii', errors='ignore')
                            print(f"✓ NMEA sentence found: {sentence}")
                            break
            
            time.sleep(1)
        except Exception as e:
            print(f"Error checking NMEA: {e}")
            break
    
    if not nmea_found:
        print("✗ No NMEA sentences detected")
        print("  The module may be outputting UBX protocol instead")
        print("  Use U-Center to configure NMEA output")
    
    bus.close()
    print("\n" + "=" * 60)
    print("DIAGNOSTIC COMPLETE")
    print("=" * 60)


# Run diagnostic
diagnose_zed_f9p(i2c_bus=1, address=0x42)

ZED-F9P I2C DIAGNOSTIC

[1] Testing I2C Bus Access...
✓ I2C Bus 1 opened successfully

[2] Testing Device Presence at 0x42...
✓ Device responds at address 0x42

[3] Reading Data Stream Registers...
✓ Registers readable
  Bytes available: 0
  ⚠ No data in buffer (might be normal if just started)

[4] Monitoring Data Stream (10 seconds)...
Timestamp | Bytes Available | Raw Data Preview
------------------------------------------------------------
00.5s    |  1776          | $GNRMC,131840.00,A,4
01.0s    |  1744          | 01636.05997,E,0.247,
01.5s    |  3488          | V*12
$GNVTG,,T,,M,0
02.0s    |  3456          | ,K,A*3A
$GNGGA,1318
02.5s    |  5202          | 1110,N,01636.05997,E
03.0s    |  5170          | 36.2,M,42.4,M,,*45

03.5s    |  6914          | 5,24,28,06,32,29,,,,
04.0s    |  6882          | ,0.92,1*06
$GNGSA,M
04.5s    |  8624          | 88,80,,,,,,,,1.12,0.
05.0s    |  8592          | 
$GNGSA,M,3,26,31,0
05.5s    | 10336          | ,,,,,1.12,0.63,0.92,
06.0s    | 10304  

In [4]:
import time
from smbus2 import SMBus
from pynmeagps import NMEAReader

bus = SMBus(1)
buffer = bytearray()
rmc_count = 0
gga_count = 0
total_sentences = 0

print("Reading for 20 seconds...")
start_time = time.time()

while time.time() - start_time < 20:
    try:
        high = bus.read_byte_data(0x42, 0xFD)
        low = bus.read_byte_data(0x42, 0xFE)
        available = (high << 8) | low
        
        # Read ANY available data (not just >100)
        if 0 < available < 0xFFFF:
            print(f"[{time.time()-start_time:.1f}s] {available} bytes available")
            
            # Read all available data
            bytes_read = 0
            while bytes_read < available:
                chunk_size = min(available - bytes_read, 32)
                raw_chunk = bus.read_i2c_block_data(0x42, 0xFF, chunk_size)
                buffer.extend(raw_chunk)
                bytes_read += chunk_size
            
            print(f"  Buffer: {len(buffer)} bytes total")
            
            # Process all complete sentences
            while b'$' in buffer and b'\n' in buffer:
                start_idx = buffer.index(b'$')
                buffer = buffer[start_idx:]  # Remove junk
                
                end_idx = buffer.index(b'\n')
                sentence = buffer[:end_idx+1]
                buffer = buffer[end_idx+1:]
                
                try:
                    sentence_str = sentence.decode('ascii', errors='ignore').strip()
                    total_sentences += 1
                    
                    print(f"  [{total_sentences}] {sentence_str[:60]}")
                    
                    if sentence_str.startswith('$') and '*' in sentence_str:
                        try:
                            parsed = NMEAReader.parse(sentence.strip())
                            
                            if parsed and hasattr(parsed, 'msgID'):
                                if parsed.msgID == 'RMC':
                                    rmc_count += 1
                                    print(f"      ✓ RMC #{rmc_count}: {parsed.lat}, {parsed.lon}")
                                elif parsed.msgID == 'GGA':
                                    gga_count += 1
                                    print(f"      ✓ GGA #{gga_count}: {parsed.lat}, {parsed.lon}")
                        except Exception as e:
                            print(f"      Parse error: {str(e)[:50]}")
                            
                except Exception as e:
                    print(f"  Decode error: {e}")
        
        time.sleep(0.2)  # Check every 0.2 seconds
        
    except Exception as e:
        print(f"I2C Error: {e}")
        time.sleep(0.1)

bus.close()
print(f"\n{'='*60}")
print(f"Total sentences: {total_sentences}")
print(f"RMC: {rmc_count}, GGA: {gga_count}")

Reading for 20 seconds...
[0.2s] 1801 bytes available
  Buffer: 1801 bytes total
  [1] $GNRMC,131544.00,A,4912.61005,N,01636.06022,E,0.148,,040126,
  [2] $GNVTG,,T,,M,0.148,N,0.275,K,A*30
  [3] $GNGGA,131544.00,4912.61005,N,01636.06022,E,1,12,0.73,236.3,
  [4] $GNGSA,M,3,25,24,28,06,32,29,,,,,,,1.25,0.73,1.01,1*08
  [5] $GNGSA,M,3,81,79,88,80,,,,,,,,,1.25,0.73,1.01,2*03
  [6] $GNGSA,M,3,26,31,06,04,23,15,,,,,,,1.25,0.73,1.01,3*0C
  [7] $GNGSA,M,3,37,11,,,,,,,,,,,1.25,0.73,1.01,4*0E
  [8] $GNGSA,M,3,,,,,,,,,,,,,1.25,0.73,1.01,5*0B
  [9] $GPGSV,2,1,07,06,19,039,26,12,52,087,14,24,14,158,15,25,82,3
  [10] $GPGSV,2,2,07,28,39,305,25,29,54,219,10,32,22,255,14,1*55
  [11] $GPGSV,2,1,05,06,19,039,18,24,14,158,15,25,82,326,23,28,39,3
  [12] $GPGSV,2,2,05,29,54,219,20,6*54
  [13] $GPGSV,1,1,03,11,40,083,,20,04,098,,31,10,316,,0*59
  [14] $GLGSV,2,1,07,69,03,243,11,70,18,289,29,78,34,109,15,79,77,0
  [15] $GLGSV,2,2,07,80,30,308,17,81,58,152,15,88,45,047,26,1*49
  [16] $GLGSV,2,1,07,69,03,243,10

In [10]:
import time
from smbus2 import SMBus
from pynmeagps import NMEAReader

bus = SMBus(1)
buffer = bytearray()
rmc_count = 0
gga_count = 0
total_sentences = 0

print("Reading for 20 seconds...")
start_time = time.time()

while time.time() - start_time < 20:
    try:
        high = bus.read_byte_data(0x42, 0xFD)
        low = bus.read_byte_data(0x42, 0xFE)
        available = (high << 8) | low
        
        if 0 < available < 0xFFFF:
            # Read all available data
            bytes_read = 0
            while bytes_read < available:
                chunk_size = min(available - bytes_read, 32)
                raw_chunk = bus.read_i2c_block_data(0x42, 0xFF, chunk_size)
                buffer.extend(raw_chunk)
                bytes_read += chunk_size
            
            # Process sentences by finding $ to $ boundaries
            while buffer.count(b'$') >= 2:  # Need at least 2 $ symbols
                # Find first $
                first_dollar = buffer.index(b'$')
                buffer = buffer[first_dollar:]  # Remove junk before
                
                # Find NEXT $ (start of next sentence)
                try:
                    next_dollar = buffer.index(b'$', 1)  # Start search after first $
                    sentence = buffer[:next_dollar]
                    buffer = buffer[next_dollar:]  # Keep from next $ onwards
                except ValueError:
                    # No second $, wait for more data
                    break
                
                try:
                    sentence_str = sentence.decode('ascii', errors='ignore').strip()
                    
                    # Must have proper NMEA format
                    if sentence_str.startswith('$') and '*' in sentence_str:
                        total_sentences += 1
                        
                        # Extract just the part before any junk
                        if '\n' in sentence_str:
                            sentence_str = sentence_str.split('\n')[0]
                        if '\r' in sentence_str:
                            sentence_str = sentence_str.split('\r')[0]
                        
                        try:
                            parsed = NMEAReader.parse(sentence_str.encode('ascii'))
                            
                            if parsed and hasattr(parsed, 'msgID'):
                                if parsed.msgID == 'RMC':
                                    rmc_count += 1
                                    print(f"✓ RMC #{rmc_count}: {parsed.lat}, {parsed.lon}")
                                elif parsed.msgID == 'GGA':
                                    gga_count += 1
                                    print(f"✓ GGA #{gga_count}: {parsed.lat}, {parsed.lon}, sats={parsed.numSV}, quality={parsed.quality}")
                        except Exception as e:
                            if total_sentences <= 10:  # Show first few errors
                                print(f"Parse error: {sentence_str[:50]} -> {e}")
                            
                except Exception as e:
                    pass
        
        time.sleep(0.2)
        
    except Exception as e:
        time.sleep(0.1)

bus.close()
print(f"\n{'='*60}")
print(f"Total sentences processed: {total_sentences}")
print(f"RMC messages: {rmc_count}")
print(f"GGA messages: {gga_count}")

Reading for 20 seconds...
✓ RMC #1: 49.210135, 16.601038
✓ GGA #1: 49.210135, 16.601038, sats=12, quality=1
✓ RMC #2: 49.2101356667, 16.601038
✓ GGA #2: 49.2101356667, 16.601038, sats=12, quality=1
✓ RMC #3: 49.2101365, 16.6010381667
✓ GGA #3: 49.2101365, 16.6010381667, sats=12, quality=1
✓ RMC #4: 49.2101358333, 16.601039
✓ GGA #4: 49.2101358333, 16.601039, sats=12, quality=1
✓ RMC #5: 49.2101346667, 16.60104
✓ GGA #5: 49.2101346667, 16.60104, sats=12, quality=1
✓ RMC #6: 49.2101346667, 16.6010403333
✓ GGA #6: 49.2101346667, 16.6010403333, sats=12, quality=1
✓ RMC #7: 49.2101355, 16.60104
✓ GGA #7: 49.2101355, 16.60104, sats=12, quality=1
✓ RMC #8: 49.2101366667, 16.6010396667
✓ GGA #8: 49.2101366667, 16.6010396667, sats=12, quality=1
✓ RMC #9: 49.2101363333, 16.60104
✓ GGA #9: 49.2101363333, 16.60104, sats=12, quality=1
✓ RMC #10: 49.210136, 16.6010406667
✓ RMC #11: 49.2101361667, 16.6010408333
✓ GGA #10: 49.2101361667, 16.6010408333, sats=12, quality=1
✓ RMC #12: 49.2101358333, 16.6